<a href="https://colab.research.google.com/github/jwork3838-cloud/MLB-hr-engine/blob/main/hr_pipeline_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pybaseball pandas numpy requests xgboost scikit-learn openpyxl gspread oauth2client

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import time
from datetime import datetime, timedelta
from pybaseball import chadwick_register
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
import os
import pickle

# Configuration
YEAR = datetime.now().year
SEASON_START = f"{YEAR}-03-01"
TODAY = datetime.now().strftime("%Y-%m-%d")
OPENWEATHER_API_KEY = "3b1c666e88254b0827f0f37e326aa46f"  # Replace with your key
BALLPARK_PAL_API_KEY = "your_ballpark_pal_key"  # Get free from ballparkpal.com

# Output file (will be used by Scriptable widget)
OUTPUT_CSV = "/content/daily_hr_predictions.csv"

In [ ]:
def get_today_games():
    url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={TODAY}"
    data = requests.get(url).json()
    games = []
    for date in data.get('dates', []):
        for game in date.get('games', []):
            if game['status']['abstractGameState'] in ['Preview', 'Live']:
                games.append({
                    'gamePk': game['gamePk'],
                    'away_team': game['teams']['away']['team']['name'],
                    'home_team': game['teams']['home']['team']['name'],
                    'away_id': game['teams']['away']['team']['id'],
                    'home_id': game['teams']['home']['team']['id'],
                    'away_probable_pitcher': game['teams']['away'].get('probablePitcher', {}).get('id'),
                    'home_probable_pitcher': game['teams']['home'].get('probablePitcher', {}).get('id')
                })
    return games

def fetch_statcast_leaderboard(min_pa=50):
    url = f"https://baseballsavant.mlb.com/leaderboard/expected_statistics?type=batter&year={YEAR}&min_pa={min_pa}&csv=true"
    df = pd.read_csv(url)
    keep = ['player_id', 'player_name', 'barrel_batted_rate', 'avg_ev', 'max_ev', 'ev50', 'launch_angle_sweet_spot_percent']
    return df[[c for c in keep if c in df.columns]]

def fetch_bat_tracking(min_pa=50):
    metrics = "k_percent,bb_percent,whiff_percent,swing_percent,zone_swing_percent,zone_swing_miss_percent,out_of_zone_swing_percent,sweet_spot_percent"
    url = f"https://baseballsavant.mlb.com/leaderboard/custom?year={YEAR}&type=batter&min_pa={min_pa}&selections={metrics}&csv=true"
    df = pd.read_csv(url)
    return df

def fetch_pitcher_arsenal(pitcher_id):
    if not pitcher_id:
        return None
    url = f"https://baseballsavant.mlb.com/leaderboard/pitch_arsenals?year={YEAR}&pitcher_type=pitcher&pid={pitcher_id}&csv=true"
    try:
        df = pd.read_csv(url)
        # Summarise: usage of four-seam, slider, etc.
        summary = df.groupby('pitch_type').agg({'usage_percent': 'first', 'avg_velo': 'first'}).to_dict()
        return summary
    except:
        return None

def fetch_platoon_splits(player_ids, season=YEAR):
    ids_str = ','.join(map(str, player_ids))
    url = f"https://statsapi.mlb.com/api/v1/people?personIds={ids_str}&hydrate=stats(group=[hitting],type=[season],sitCodes=[vr,vl],season={season})"
    data = requests.get(url).json()
    rows = []
    for person in data.get('people', []):
        pid = person['id']
        name = person['fullName']
        for stat_entry in person.get('stats', []):
            for split in stat_entry.get('splits', []):
                sit = split.get('split', {}).get('description', '')
                stat = split.get('stat', {})
                rows.append({
                    'player_id': pid,
                    'name': name,
                    'situation': sit,
                    'hr': stat.get('homeRuns', 0),
                    'ab': stat.get('atBats', 0),
                    'bb': stat.get('baseOnBalls', 0),
                    'so': stat.get('strikeOuts', 0),
                    'avg': stat.get('avg', 0),
                    'slg': stat.get('slugging', 0)
                })
    return pd.DataFrame(rows)

def fetch_weather(lat, lon):
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={OPENWEATHER_API_KEY}&units=imperial"
    resp = requests.get(url).json()
    if 'main' in resp:
        return {
            'temp': resp['main']['temp'],
            'wind_speed': resp['wind']['speed'],
            'wind_deg': resp['wind']['deg']
        }
    return None

# Stadium coordinates (same as your Scriptable script)
stadium_coords = {
    "Arizona Diamondbacks": (33.4453, -112.0667),
    "Atlanta Braves": (33.8907, -84.4677),
    "Baltimore Orioles": (39.2838, -76.6218),
    "Boston Red Sox": (42.3467, -71.0972),
    "Chicago Cubs": (41.9484, -87.6553),
    "Chicago White Sox": (41.8299, -87.6338),
    "Cincinnati Reds": (39.0979, -84.5082),
    "Cleveland Guardians": (41.4962, -81.6852),
    "Colorado Rockies": (39.7559, -104.9942),
    "Detroit Tigers": (42.3390, -83.0485),
    "Houston Astros": (29.7573, -95.3555),
    "Kansas City Royals": (39.0517, -94.4803),
    "Los Angeles Angels": (33.8003, -117.8827),
    "Los Angeles Dodgers": (34.0739, -118.2400),
    "Miami Marlins": (25.7781, -80.2197),
    "Milwaukee Brewers": (43.0280, -87.9712),
    "Minnesota Twins": (44.9817, -93.2776),
    "New York Mets": (40.7571, -73.8458),
    "New York Yankees": (40.8296, -73.9262),
    "Oakland Athletics": (38.5802, -121.4687),
    "Philadelphia Phillies": (39.9061, -75.1665),
    "Pittsburgh Pirates": (40.4469, -80.0057),
    "San Diego Padres": (32.7076, -117.1570),
    "San Francisco Giants": (37.7786, -122.3893),
    "Seattle Mariners": (47.5914, -122.3325),
    "St. Louis Cardinals": (38.6226, -90.1928),
    "Tampa Bay Rays": (27.7682, -82.6534),
    "Texas Rangers": (32.7512, -97.0832),
    "Toronto Blue Jays": (43.6414, -79.3894),
    "Washington Nationals": (38.8730, -77.0074)
}

In [ ]:
def build_feature_matrix(games, statcast_df, bat_track_df, splits_df, id_map):
    # This is a simplified version – you would expand it with all features from the PDF
    # For each batter on each team (active roster), combine features
    all_rows = []
    for game in games:
        home_team = game['home_team']
        away_team = game['away_team']
        home_pitcher_id = game['home_probable_pitcher']
        away_pitcher_id = game['away_probable_pitcher']
        # Get weather for home park
        coords = stadium_coords.get(home_team, (None, None))
        weather = fetch_weather(coords[0], coords[1]) if coords[0] else None

        # Fetch active batters for both teams (use MLB API)
        for team in [home_team, away_team]:
            team_id = game['home_id'] if team == home_team else game['away_id']
            batters_url = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/roster?rosterType=active&season={YEAR}"
            batters = requests.get(batters_url).json().get('roster', [])
            for batter in batters:
                player_id = batter['person']['id']
                name = batter['person']['fullName']
                # Merge statcast
                sc = statcast_df[statcast_df['player_id'] == player_id]
                if sc.empty:
                    continue
                bt = bat_track_df[bat_track_df['player_id'] == player_id]
                splits = splits_df[splits_df['player_id'] == player_id]
                # Platoon: which side pitcher?
                pitcher_hand = "R"  # would need to fetch from pitcher's profile
                batter_hand = "R"   # same
                platoon_adv = 1 if (batter_hand == 'L' and pitcher_hand == 'R') or (batter_hand == 'R' and pitcher_hand == 'L') else 0

                row = {
                    'player_name': name,
                    'team': team,
                    'opponent_pitcher_id': home_pitcher_id if team == away_team else away_pitcher_id,
                    'barrel_rate': sc.iloc[0]['barrel_batted_rate'] if 'barrel_batted_rate' in sc else 0,
                    'avg_ev': sc.iloc[0]['avg_ev'] if 'avg_ev' in sc else 85,
                    'ev50': sc.iloc[0]['ev50'] if 'ev50' in sc else 85,
                    'sweet_spot_pct': sc.iloc[0]['launch_angle_sweet_spot_percent'] if 'launch_angle_sweet_spot_percent' in sc else 0,
                    'whiff_pct': bt.iloc[0]['whiff_percent'] if not bt.empty and 'whiff_percent' in bt else 25,
                    'chase_pct': bt.iloc[0]['out_of_zone_swing_percent'] if not bt.empty and 'out_of_zone_swing_percent' in bt else 30,
                    'platoon_advantage': platoon_adv,
                    'temp': weather['temp'] if weather else 70,
                    'wind_speed': weather['wind_speed'] if weather else 0,
                }
                # Add park factor (static)
                park_factor = 1.0  # you can load from a CSV or FanGraphs
                row['park_factor'] = park_factor
                all_rows.append(row)
    return pd.DataFrame(all_rows)

In [ ]:
import pandas as pd
import xgboost as xgb
from pybaseball import statcast
from sklearn.model_selection import train_test_split
import numpy as np
import os

# --- Configuration & Setup ---
model_filename = 'hr_model.json'

def train_and_save_model():
    # 1. Fetch real data from Statcast
    # Using a 30-day window from the 2026 season for training
    print("Fetching live Statcast data for model training...")
    data = statcast(start_dt='2026-05-01', end_dt='2026-05-30')

    # 2. Preprocess Data
    # Identify home runs and clean features
    data['is_hr'] = (data['events'] == 'home_run').astype(int)
    features = ['launch_speed', 'launch_angle', 'release_speed', 'plate_x', 'plate_z']

    # Drop rows where critical Statcast data is missing
    clean_data = data.dropna(subset=features + ['is_hr'])

    X = clean_data[features]
    y = clean_data['is_hr']

    # 3. Train Model
    print(f"Training on {len(X)} events...")
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        eval_metric='logloss'
    )
    model.fit(X, y)

    # 4. Save Model
    model.save_model(model_filename)
    print(f"Model successfully trained and saved to '{model_filename}'")
    return model

# --- Execution ---
if os.path.exists(model_filename):
    print("Loading existing model...")
    model = xgb.XGBClassifier()
    model.load_model(model_filename)
else:
    print("No existing model found. Training from scratch...")
    model = train_and_save_model()


In [ ]:
import urllib.request
import requests
import time

# ==============================================================================
# NETWORK PATCH: Bypass WAF / 403 Forbidden in Google Colab
# ==============================================================================

# 1. Patch urllib (Used heavily by pandas.read_csv and pybaseball internally)
opener = urllib.request.build_opener()
opener.addheaders = [
    ('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'),
    ('Accept', 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'),
    ('Accept-Language', 'en-US,en;q=0.5'),
    ('Connection', 'keep-alive')
]
urllib.request.install_opener(opener)

# 2. Patch requests (Used by other pybaseball modules)
original_get = requests.get
original_post = requests.post

def patched_get(*args, **kwargs):
    headers = kwargs.pop('headers', {})
    headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'
    })
    kwargs['headers'] = headers
    return original_get(*args, **kwargs)

def patched_post(*args, **kwargs):
    headers = kwargs.pop('headers', {})
    headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8'
    })
    kwargs['headers'] = headers
    return original_post(*args, **kwargs)

requests.get = patched_get
requests.post = patched_post

# ==============================================================================
# INFERENCE PIPELINE
# ==============================================================================

def run_daily_inference():
    print("Checking today's slate...")
    games = get_today_games()
    if not games:
        print("No games today.")
        return

    # Adding slight sleep delays to prevent hitting rate limits during the batch pull
    print("Fetching statcast leaderboard...")
    statcast_df = fetch_statcast_leaderboard()
    time.sleep(1)

    print("Fetching bat tracking data...")
    bat_track_df = fetch_bat_tracking()
    time.sleep(1)

    all_player_ids = set(statcast_df['player_id'].unique()) | set(bat_track_df['player_id'].unique())

    print("Fetching platoon splits...")
    splits_df = fetch_platoon_splits(list(all_player_ids))
    time.sleep(1)

    print("Pulling Chadwick register...")
    id_map = chadwick_register()[['key_mlbam', 'key_fangraphs']].dropna()

    print("Building feature matrix...")
    feature_df = build_feature_matrix(games, statcast_df, bat_track_df, splits_df, id_map)

    # Select only the features used in training
    feature_cols = [
        'barrel_rate', 'avg_ev', 'ev50', 'sweet_spot_pct', 'whiff_pct',
        'chase_pct', 'platoon_advantage', 'temp', 'wind_speed', 'park_factor'
    ]
    X = feature_df[feature_cols].fillna(0)

    print("Executing model inferences...")
    probs = model.predict_proba(X)[:, 1]

    feature_df['hr_prob'] = probs

    # Sort by probability
    top = feature_df.sort_values('hr_prob', ascending=False).head(50)
    top.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved {len(top)} predictions to {OUTPUT_CSV}")

run_daily_inference()


In [ ]:
# Cell 7 – Upload CSV to Google Drive (public link)
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Create destination folder in Drive
dest_folder = '/content/drive/MyDrive/MLB_HR_Predictions/'
os.makedirs(dest_folder, exist_ok=True)

# Copy CSV to Drive
shutil.copy('/content/daily_hr_predictions.csv', dest_folder + 'daily_hr_predictions.csv')
print(f"CSV copied to: {dest_folder}")

# Generate a shareable link (you will need to set permission to "Anyone with the link" manually)
print("\n👉 Now go to https://drive.google.com")
print("   Navigate to 'My Drive/MLB_HR_Predictions/'")
print("   Right‑click 'daily_hr_predictions.csv' → 'Get link'")
print("   Change general access to 'Anyone with the link'")
print("   Copy the link and paste it into the Scriptable widget's CSV_URL variable.")